In [ ]:
# Databricks notebook source
# DBTITLE 1,SILVER MEMBER GROUP ENROLLMENT PIPELINE
import logging
from datetime import datetime
import os
from pyspark.sql.functions import col, sha2, concat_ws, current_timestamp, to_date

In [ ]:
# Ensure target databases exist under claimspan catalog
spark.sql("CREATE DATABASE IF NOT EXISTS claimspan.silver")
spark.sql("CREATE DATABASE IF NOT EXISTS claimspan.gold")
silverMemGroupTable = "claimspan.silver.silver_member_group"
print(f"Target Silver Member Group Table: {silverMemGroupTable}")


In [ ]:
# Ensure target silver table exists via DDL fallback
spark.sql("""
CREATE TABLE IF NOT EXISTS claimspan.silver.silver_member_group (
    SubscriberID        STRING,
    BeneficiaryID       STRING,
    CMSContractNumber   STRING,
    GroupNumber         STRING,
    GroupSuffix         STRING,
    StartDate           DATE,
    EndDate             DATE,
    SourceFileID        BIGINT,
    LoadDateTime        TIMESTAMP,
    hashKey             STRING
) USING delta;
""")
if spark.catalog.tableExists("claimspan.silver.silver_member"):
    df_src = spark.table("claimspan.silver.silver_member")
    
    df_grp = df_src.select(
        col("SubscriberID"),
        col("BeneficiaryID"),
        col("ProductID").alias("CMSContractNumber"),
        col("PlanMemberID").alias("GroupNumber"),
        col("ClientID").alias("GroupSuffix"),
        to_date(col("LoadDateTime")).alias("StartDate"),
        to_date(col("DeceasedDate")).alias("EndDate"),
        col("FileID").cast("bigint").alias("SourceFileID"),
        current_timestamp().alias("LoadDateTime")
    ).distinct()
    
    df_grp_final = df_grp.withColumn(
        "hashKey",
        sha2(concat_ws("|", col("SubscriberID"), col("BeneficiaryID"), col("CMSContractNumber"), col("GroupNumber"), col("GroupSuffix")), 256)
    )
    
    df_grp_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silverMemGroupTable)
    print(f"=== Silver MemberGroup processing complete: {df_grp_final.count()} records created ===")
else:
    print("Silver member table not found, waiting for member ingestion.")